<a href="https://colab.research.google.com/github/lam36211-bit/reit-sector-attribution-dashboard/blob/main/reit_sector_attribution_dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# REIT Sector Performance & Rate-Sensitivity Dashboard

**Institutional-style performance reporting for a REIT portfolio** — returns, risk-adjusted metrics, drawdown analytics, rolling diagnostics, sector contribution-to-return, and Brinson–Fachler attribution *by property sector*, delivered as a standalone interactive Plotly dashboard.

Built for a real-estate-focused analyst: the portfolio holds ten best-in-class REITs across eight property sectors (industrial, data centers, towers, residential, retail, office, healthcare, storage), benchmarked against the REIT index (VNQ), with a dedicated rate-sensitivity view plotting the rolling correlation between REITs and long Treasuries through the 2022 rate shock.

| | |
|---|---|
| **Universe** | 10 REITs across 8 property sectors + VNQ / SPY / TLT context |
| **History** | 10 years of daily adjusted closes (Yahoo Finance) |
| **Risk-free** | FRED `DGS3MO` with automatic `^IRX` fallback |
| **Benchmark** | VNQ (REIT index) for performance & attribution; SPY for market beta |
| **Star chart** | Rolling VNQ–TLT correlation (REITs' rate sensitivity) |
| **Output** | Tabbed `dashboard.html` + CSV exports |

**Run order:** every cell top-to-bottom. Runtime ≈ 60–90 seconds on a free Colab instance.

## 0 · Environment Setup

Installs pinned-free versions of the data and plotting stack. `fredapi` is optional — the notebook falls back to Yahoo's `^IRX` T-bill yield if no FRED API key is provided.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [1]:
# ── Cell 0: Install dependencies ─────────────────────────────────────────────
# -q keeps Colab output clean. Re-running is safe (pip no-ops if satisfied).
!pip install -q yfinance fredapi plotly pandas numpy

print("Environment ready.")

Environment ready.


## 1 · Portfolio Configuration

A conviction-weighted portfolio of ten REITs, one or two best-in-class names per property sector, with a deliberate tilt toward structural winners (industrial, data centers, towers) and a retained underweight to the challenged sectors (office, malls) — kept in the book *because* their drawdowns make the attribution story real.

Three design choices worth noting:

1. **Property-sector map** — the "asset class" here is the REIT property sector. Brinson decomposes active return *by sector*, which is exactly how a real-estate PM thinks: industrial vs. office, not ticker vs. ticker.
2. **Benchmark = VNQ** — the REIT index is the natural benchmark for a REIT book (SPY would just measure "REITs vs. stocks"). SPY and TLT are carried as *context* series for market beta and the rate-sensitivity chart.
3. **Policy benchmark weights** — Brinson needs a benchmark with sector weights. We use an equal-sector-weight neutral policy so the allocation effect cleanly isolates the thesis tilts (swapping in VNQ's published sector weights is a documented upgrade).

In [2]:
# ── Cell 1: Imports & configuration ──────────────────────────────────────────
from __future__ import annotations

import time
import warnings
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
import yfinance as yf
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

warnings.filterwarnings("ignore", category=FutureWarning)
pio.templates.default = "plotly_white"

# ------------------------------------------------------------------ constants
TRADING_DAYS: int = 252          # annualization convention for daily data
ROLL_VOL_WINDOW: int = 90        # rolling volatility window (days)
ROLL_SHARPE_WINDOW: int = 252    # rolling Sharpe window (days)
VAR_LEVEL: float = 0.95          # confidence level for historical VaR / CVaR
MAX_FFILL_DAYS: int = 5          # longest gap we will forward-fill
FRED_API_KEY: str = ""           # optional — leave blank to use ^IRX fallback

# ------------------------------------------------------------------ portfolio
@dataclass(frozen=True)
class PortfolioConfig:
    """Immutable container for the REIT portfolio definition."""
    weights: dict[str, float] = field(default_factory=lambda: {
        # Structural winners (overweight) --------------------------- 38%
        "PLD":  0.15,   # Prologis — industrial / logistics (e-commerce warehouses)
        "EQIX": 0.13,   # Equinix — data centers (AI infrastructure)
        "AMT":  0.10,   # American Tower — cell towers
        # Core / market-weight -------------------------------------- 46%
        "AVB":  0.08,   # AvalonBay — coastal multifamily
        "MAA":  0.09,   # Mid-America — Sunbelt multifamily
        "O":    0.10,   # Realty Income — net-lease retail (Weitzman-adjacent)
        "WELL": 0.10,   # Welltower — senior housing / healthcare
        "EXR":  0.09,   # Extra Space — self-storage
        # Challenged sectors (retained for the attribution story) --- 16%
        "SPG":  0.08,   # Simon Property — Class A malls
        "BXP":  0.08,   # BXP — Class A office (the drawdown that makes charts interesting)
    })
    property_sector: dict[str, str] = field(default_factory=lambda: {
        "PLD": "Industrial", "EQIX": "Data Centers", "AMT": "Towers",
        "AVB": "Residential", "MAA": "Residential",
        "O": "Retail", "SPG": "Retail",
        "WELL": "Healthcare", "EXR": "Storage", "BXP": "Office",
    })
    # Neutral policy benchmark: equal weight across the 8 property sectors,
    # so the allocation effect isolates the portfolio's sector tilts.
    benchmark_ticker: str = "VNQ"                 # REIT index — headline benchmark
    context_tickers: tuple[str, ...] = ("SPY", "TLT")  # market beta + rate sensitivity
    years: int = 10

    @property
    def asset_class(self) -> dict[str, str]:      # alias kept for the metrics/Brinson code
        return self.property_sector

    @property
    def benchmark_class_weights(self) -> dict[str, float]:
        sectors = sorted(set(self.property_sector.values()))
        return {s: 1.0 / len(sectors) for s in sectors}   # equal-sector-weight neutral

    def validate(self) -> None:
        """Fail fast on malformed config instead of producing silent garbage."""
        w = sum(self.weights.values())
        if not np.isclose(w, 1.0, atol=1e-9):
            raise ValueError(f"Portfolio weights sum to {w:.6f}, expected 1.0")
        missing = set(self.weights) - set(self.property_sector)
        if missing:
            raise ValueError(f"Tickers missing a property-sector label: {missing}")

CFG = PortfolioConfig()
CFG.validate()

END_DATE = pd.Timestamp.today().normalize()
START_DATE = END_DATE - pd.DateOffset(years=CFG.years)
ALL_TICKERS = sorted(set(CFG.weights) | {CFG.benchmark_ticker} | set(CFG.context_tickers))

print(f"REITs    : {list(CFG.weights)}")
print(f"Context  : benchmark={CFG.benchmark_ticker}, series={list(CFG.context_tickers)}")
print(f"Sectors  : {sorted(set(CFG.property_sector.values()))}")
print(f"Window   : {START_DATE.date()} -> {END_DATE.date()}")
print(f"Weights OK — sum = {sum(CFG.weights.values()):.2f}")

REITs    : ['PLD', 'EQIX', 'AMT', 'AVB', 'MAA', 'O', 'WELL', 'EXR', 'SPG', 'BXP']
Context  : benchmark=VNQ, series=['SPY', 'TLT']
Sectors  : ['Data Centers', 'Healthcare', 'Industrial', 'Office', 'Residential', 'Retail', 'Storage', 'Towers']
Window   : 2016-07-08 -> 2026-07-08
Weights OK — sum = 1.00


## 2 · Data Collection Pipeline

**Prices:** batch-downloaded from Yahoo Finance with `auto_adjust=True` (splits and dividends folded into the close — the correct series for total-return math), wrapped in exponential-backoff retries because Yahoo intermittently rate-limits Colab IPs.

**Risk-free rate:** a three-stage fallback chain: FRED `DGS3MO` → Yahoo `^IRX` (13-week T-bill) → 4% constant. Annualized percent yields are converted to a **daily decimal rate** (`y / 100 / 252`) and aligned to the trading calendar.

In [3]:
# ── Cell 2: Price download with retry & validation ───────────────────────────
def fetch_prices(
    tickers: list[str],
    start: pd.Timestamp,
    end: pd.Timestamp,
    retries: int = 3,
    pause: float = 2.0,
) -> pd.DataFrame:
    """Download adjusted close prices for `tickers`.

    Returns a DataFrame indexed by date with one column per ticker.
    Raises RuntimeError if all retries fail or no data comes back.
    """
    last_err: Exception | None = None
    for attempt in range(1, retries + 1):
        try:
            raw = yf.download(
                tickers, start=start, end=end,
                auto_adjust=True, progress=False, threads=True,
            )
            prices = raw["Close"] if isinstance(raw.columns, pd.MultiIndex) else raw[["Close"]]
            if isinstance(prices, pd.Series):
                prices = prices.to_frame(tickers[0])
            prices = prices.dropna(how="all")
            if prices.empty:
                raise RuntimeError("Yahoo returned an empty frame")
            return prices[sorted(prices.columns)]
        except Exception as err:                       # noqa: BLE001 — network layer
            last_err = err
            wait = pause * (2 ** (attempt - 1))
            print(f"  attempt {attempt}/{retries} failed ({err}); retrying in {wait:.0f}s")
            time.sleep(wait)
    raise RuntimeError(f"Price download failed after {retries} attempts: {last_err}")


prices_raw = fetch_prices(ALL_TICKERS, START_DATE, END_DATE)

# Coverage report: flag tickers with thin history (e.g. recently listed ETFs)
expected = len(prices_raw)
coverage = prices_raw.notna().sum().div(expected).sort_values()
print(f"Downloaded {prices_raw.shape[0]:,} rows x {prices_raw.shape[1]} tickers")
thin = coverage[coverage < 0.95]
print("Coverage < 95%:", dict(thin.round(3)) if not thin.empty else "none — all tickers healthy")

Downloaded 2,512 rows x 13 tickers
Coverage < 95%: none — all tickers healthy


In [4]:
# ── Cell 3: Risk-free rate (FRED -> ^IRX -> constant fallback) ───────────────
def fetch_risk_free_daily(
    index: pd.DatetimeIndex,
    fred_key: str = "",
    fallback_annual: float = 0.04,
) -> pd.Series:
    """Return the daily decimal risk-free rate aligned to `index`.

    Sources, in order of preference:
      1. FRED DGS3MO (3-month T-bill, annualized %)   — needs an API key
      2. Yahoo ^IRX (13-week T-bill yield, annualized %)
      3. Constant `fallback_annual`
    """
    annual_pct: pd.Series | None = None

    if fred_key:
        try:
            from fredapi import Fred
            annual_pct = Fred(api_key=fred_key).get_series("DGS3MO")
            print("Risk-free source: FRED DGS3MO")
        except Exception as err:                       # noqa: BLE001
            print(f"FRED unavailable ({err}); falling back to ^IRX")

    if annual_pct is None:
        try:
            irx = yf.download("^IRX", start=index[0], end=index[-1],
                              auto_adjust=True, progress=False)["Close"]
            annual_pct = irx.squeeze()
            print("Risk-free source: Yahoo ^IRX (13-week T-bill)")
        except Exception as err:                       # noqa: BLE001
            print(f"^IRX unavailable ({err}); using {fallback_annual:.1%} constant")

    if annual_pct is None or annual_pct.dropna().empty:
        return pd.Series(fallback_annual / TRADING_DAYS, index=index, name="rf_daily")

    daily = (annual_pct / 100.0 / TRADING_DAYS)
    daily = daily.reindex(index).ffill().bfill()       # align to trading calendar
    return daily.rename("rf_daily")


rf_daily = fetch_risk_free_daily(prices_raw.index, FRED_API_KEY)
print(f"Current annualized risk-free ~ {rf_daily.iloc[-1] * TRADING_DAYS:.2%}")

Risk-free source: Yahoo ^IRX (13-week T-bill)
Current annualized risk-free ~ 3.67%


## 3 · Data Cleaning & Feature Engineering

Cross-asset ETFs trade on slightly different calendars (bond-market holidays, etc.), so we:

1. forward-fill gaps up to 5 days (a stale price is correct for an untraded asset — its return that day is zero),
2. drop the leading window until **every** ticker has data, and
3. assert there are no remaining NaNs before any math runs.

**The key engineering decision:** we compute **both** log and simple returns.
Log returns are additive **through time** (`Σ log-returns = log of total growth`) — ideal for statistics.
Simple returns are additive **across assets** (`r_p = Σ wᵢ·rᵢ`) — required for portfolio construction and attribution. Mixing these up is the single most common quant-intern bug.

In [5]:
# ── Cell 4: Clean & validate ─────────────────────────────────────────────────
def clean_prices(prices: pd.DataFrame, max_ffill: int = MAX_FFILL_DAYS) -> pd.DataFrame:
    """Forward-fill short gaps, trim the warm-up window, and hard-validate."""
    out = prices.ffill(limit=max_ffill)
    first_full = out.dropna().index.min()              # first day all tickers exist
    out = out.loc[first_full:]
    if out.isna().any().any():
        bad = out.columns[out.isna().any()].tolist()
        raise ValueError(f"Unfillable gaps remain in: {bad}")
    return out


prices = clean_prices(prices_raw)
print(f"Clean panel: {prices.shape[0]:,} days x {prices.shape[1]} tickers "
      f"({prices.index[0].date()} -> {prices.index[-1].date()})")

Clean panel: 2,512 days x 13 tickers (2016-07-08 -> 2026-07-07)


In [6]:
# ── Cell 5: Returns engine ───────────────────────────────────────────────────
def to_log_returns(prices: pd.DataFrame) -> pd.DataFrame:
    """Daily log returns: ln(P_t / P_{t-1}). Additive through time."""
    return np.log(prices / prices.shift(1)).dropna()


def to_simple_returns(prices: pd.DataFrame) -> pd.DataFrame:
    """Daily simple returns: P_t / P_{t-1} - 1. Additive across assets."""
    return prices.pct_change().dropna()


log_rets    = to_log_returns(prices)
simple_rets = to_simple_returns(prices)
rf_daily    = rf_daily.reindex(simple_rets.index).ffill()

PORT_TICKERS = list(CFG.weights)                      # 10 REITs — the portfolio
W = pd.Series(CFG.weights)                             # static target weights

# Daily-rebalanced portfolio return (industry default for reporting):
port_ret  = simple_rets[PORT_TICKERS].mul(W, axis=1).sum(axis=1).rename("Portfolio")
bench_ret = simple_rets[CFG.benchmark_ticker].rename("Benchmark")   # VNQ
spy_ret   = simple_rets["SPY"].rename("SPY")          # market context
tlt_ret   = simple_rets["TLT"].rename("TLT")          # rate context

# Wealth indices (growth of $10,000)
wealth_port  = 10_000 * (1 + port_ret).cumprod()
wealth_bench = 10_000 * (1 + bench_ret).cumprod()
wealth_spy   = 10_000 * (1 + spy_ret).cumprod()

print(f"REIT portfolio total return : {wealth_port.iloc[-1] / 10_000 - 1:+.1%}")
print(f"VNQ benchmark total return  : {wealth_bench.iloc[-1] / 10_000 - 1:+.1%}")
print(f"SPY (market) total return   : {wealth_spy.iloc[-1] / 10_000 - 1:+.1%}")

REIT portfolio total return : +161.5%
VNQ benchmark total return  : +62.9%
SPY (market) total return   : +312.8%


## 4 · Metrics Engine

Every function below is pure (Series in → float out), documented, and uses standard institutional conventions:

| Metric | Formula |
|---|---|
| CAGR | $(W_T / W_0)^{252/T} - 1$ |
| Ann. volatility | $\sigma_{daily} \cdot \sqrt{252}$ |
| Sharpe | $\bar{r}_{excess} / \sigma \cdot \sqrt{252}$ |
| Sortino | Sharpe with **downside** deviation in the denominator |
| Max drawdown | $\min_t \left( W_t / \max_{s \le t} W_s - 1 \right)$ |
| Calmar | CAGR / \|MaxDD\| |
| VaR / CVaR (95%) | 5th percentile of daily returns / mean of returns below it |

In [7]:
# ── Cell 6: Risk & performance metrics ───────────────────────────────────────
def cagr(returns: pd.Series) -> float:
    """Compound annual growth rate from a daily simple-return series."""
    total_growth = float((1 + returns).prod())
    years = len(returns) / TRADING_DAYS
    return total_growth ** (1 / years) - 1


def ann_vol(returns: pd.Series) -> float:
    """Annualized volatility via the sqrt-252 rule."""
    return float(returns.std(ddof=1)) * np.sqrt(TRADING_DAYS)


def sharpe(returns: pd.Series, rf: pd.Series) -> float:
    """Annualized Sharpe ratio on daily excess returns."""
    excess = returns - rf.reindex(returns.index).ffill()
    sd = excess.std(ddof=1)
    return float(excess.mean() / sd) * np.sqrt(TRADING_DAYS) if sd > 0 else np.nan


def sortino(returns: pd.Series, rf: pd.Series) -> float:
    """Sortino ratio — penalizes only downside deviation."""
    excess = returns - rf.reindex(returns.index).ffill()
    downside = excess[excess < 0]
    dd = np.sqrt((downside ** 2).mean())               # root-mean-square shortfall
    return float(excess.mean() / dd) * np.sqrt(TRADING_DAYS) if dd > 0 else np.nan


def drawdown_series(returns: pd.Series) -> pd.Series:
    """Percentage drawdown from the running peak, at every date."""
    wealth = (1 + returns).cumprod()
    return wealth / wealth.cummax() - 1


def max_drawdown(returns: pd.Series) -> tuple[float, pd.Timestamp, pd.Timestamp]:
    """(max drawdown, peak date, trough date)."""
    dd = drawdown_series(returns)
    trough = dd.idxmin()
    wealth = (1 + returns).cumprod()
    peak = wealth.loc[:trough].idxmax()
    return float(dd.min()), peak, trough


def calmar(returns: pd.Series) -> float:
    """CAGR per unit of max drawdown — the tail-risk-adjusted return."""
    mdd, _, _ = max_drawdown(returns)
    return cagr(returns) / abs(mdd) if mdd < 0 else np.nan


def var_cvar(returns: pd.Series, level: float = VAR_LEVEL) -> tuple[float, float]:
    """Historical daily VaR and CVaR at `level` confidence."""
    var = float(returns.quantile(1 - level))
    cvar = float(returns[returns <= var].mean())
    return var, cvar


def beta(returns: pd.Series, market: pd.Series) -> float:
    """OLS beta of `returns` on the market series."""
    aligned = pd.concat([returns, market], axis=1).dropna()
    cov = aligned.cov().iloc[0, 1]
    return float(cov / aligned.iloc[:, 1].var(ddof=1))


def correlation(returns: pd.Series, other: pd.Series) -> float:
    """Pairwise Pearson correlation of daily returns."""
    return float(pd.concat([returns, other], axis=1).dropna().corr().iloc[0, 1])


def metric_row(returns: pd.Series, rf: pd.Series,
               market: pd.Series, rates: pd.Series) -> dict:
    """Full metric suite for one return stream.

    `market` (SPY) drives beta; `rates` (TLT) drives the rate-sensitivity
    correlation that matters for a REIT book.
    """
    mdd, peak, trough = max_drawdown(returns)
    var, cvar = var_cvar(returns)
    return {
        "Total Return":  float((1 + returns).prod() - 1),
        "CAGR":          cagr(returns),
        "Ann. Vol":      ann_vol(returns),
        "Sharpe":        sharpe(returns, rf),
        "Sortino":       sortino(returns, rf),
        "Max DD":        mdd,
        "DD Peak":       peak.date(),
        "DD Trough":     trough.date(),
        "Calmar":        calmar(returns),
        "VaR 95% (d)":   var,
        "CVaR 95% (d)":  cvar,
        "Hit Rate":      float((returns > 0).mean()),
        "Beta vs SPY":   beta(returns, market),
        "Corr vs TLT":   correlation(returns, rates),
    }


rows = {"REIT Portfolio":  metric_row(port_ret, rf_daily, spy_ret, tlt_ret),
        "Benchmark (VNQ)": metric_row(bench_ret, rf_daily, spy_ret, tlt_ret)}
for t in PORT_TICKERS:
    rows[t] = metric_row(simple_rets[t], rf_daily, spy_ret, tlt_ret)

metrics_df = pd.DataFrame(rows).T
pct_cols = ["Total Return", "CAGR", "Ann. Vol", "Max DD", "VaR 95% (d)", "CVaR 95% (d)", "Hit Rate"]
display(metrics_df.style
        .format({c: "{:.2%}" for c in pct_cols} |
                {c: "{:.2f}" for c in ["Sharpe", "Sortino", "Calmar", "Beta vs SPY", "Corr vs TLT"]})
        .background_gradient(subset=["Sharpe"], cmap="RdYlGn"))

,Total Return,CAGR,Ann. Vol,Sharpe,Sortino,Max DD,DD Peak,DD Trough,Calmar,VaR 95% (d),CVaR 95% (d),Hit Rate,Beta vs SPY,Corr vs TLT
REIT Portfolio,161.50%,10.13%,21.67%,0.45,0.42,-40.42%,2020-02-18,2020-03-23,0.25,-1.96%,-3.15%,54.00%,0.82,0.06
Benchmark (VNQ),62.90%,5.02%,20.74%,0.23,0.21,-42.40%,2020-02-21,2020-03-23,0.12,-1.88%,-3.05%,53.21%,0.83,0.05
PLD,284.88%,14.48%,27.02%,0.55,0.53,-43.30%,2022-04-28,2025-04-08,0.33,-2.52%,-3.97%,53.33%,0.96,0.06
EQIX,218.64%,12.33%,27.20%,0.48,0.48,-41.77%,2021-09-03,2022-10-14,0.30,-2.62%,-3.86%,52.93%,0.81,0.09
AMT,87.56%,6.52%,26.33%,0.28,0.28,-45.34%,2021-09-08,2023-10-04,0.14,-2.63%,-3.73%,52.37%,0.65,0.11
AVB,48.49%,4.05%,24.72%,0.19,0.19,-46.91%,2020-02-18,2020-03-23,0.09,-2.27%,-3.50%,52.33%,0.78,-0.01
MAA,87.06%,6.49%,24.17%,0.29,0.27,-45.40%,2021-12-31,2023-10-30,0.14,-2.28%,-3.44%,52.25%,0.73,0.03
O,51.61%,4.27%,25.65%,0.20,0.19,-48.28%,2020-02-21,2020-03-18,0.09,-2.18%,-3.64%,52.73%,0.70,0.08
WELL,351.75%,16.34%,31.95%,0.56,0.55,-63.33%,2019-09-04,2020-03-18,0.26,-2.56%,-4.45%,54.48%,0.82,0.04
EXR,131.50%,8.79%,26.93%,0.36,0.35,-51.36%,2021-12-31,2023-10-25,0.17,-2.51%,-4.02%,52.49%,0.65,0.09


## 5 · Rolling Diagnostics

Point-in-time metrics hide regime changes. Rolling windows reveal them:

- **90-day rolling volatility** — spot risk-regime shifts (COVID 2020, the 2022 rate shock).
- **252-day rolling Sharpe** — was risk-adjusted performance persistent, or one lucky year?

In [8]:
# ── Cell 7: Rolling volatility & rolling Sharpe ──────────────────────────────
def rolling_vol(returns: pd.Series, window: int = ROLL_VOL_WINDOW) -> pd.Series:
    """Rolling annualized volatility."""
    return returns.rolling(window).std(ddof=1) * np.sqrt(TRADING_DAYS)


def rolling_sharpe(returns: pd.Series, rf: pd.Series,
                   window: int = ROLL_SHARPE_WINDOW) -> pd.Series:
    """Rolling annualized Sharpe ratio."""
    excess = returns - rf.reindex(returns.index).ffill()
    mu = excess.rolling(window).mean()
    sd = excess.rolling(window).std(ddof=1)
    return (mu / sd) * np.sqrt(TRADING_DAYS)


roll_vol_port     = rolling_vol(port_ret).dropna()
roll_vol_bench    = rolling_vol(bench_ret).dropna()
roll_sharpe_port  = rolling_sharpe(port_ret, rf_daily).dropna()
roll_sharpe_bench = rolling_sharpe(bench_ret, rf_daily).dropna()

print(f"Latest 90d vol   — portfolio {roll_vol_port.iloc[-1]:.1%} | SPY {roll_vol_bench.iloc[-1]:.1%}")
print(f"Latest 1y Sharpe — portfolio {roll_sharpe_port.iloc[-1]:.2f} | SPY {roll_sharpe_bench.iloc[-1]:.2f}")

Latest 90d vol   — portfolio 15.8% | SPY 15.6%
Latest 1y Sharpe — portfolio 0.96 | SPY 0.72


## 6 · Performance Attribution

Two complementary decompositions — this is the analytical core of the project.

**(a) Contribution to return.** For a daily-rebalanced portfolio, asset *i*'s raw contribution is
$c_i = \sum_t w_i \, r_{i,t}$. Raw contributions sum to the *arithmetic* total, not the *geometric* (compounded) total, so we proportionally scale them to close the gap — a transparent, simple alternative to Carino logarithmic linking (listed as an upgrade).

**(b) Brinson–Fachler.** Decomposes active return vs. an equal-sector-weight policy benchmark, **per property sector**:

- **Allocation** $= (w_p - w_b)(r_{b,sector} - r_b)$ — did over/underweighting *sectors* help? (Did overweighting industrial and underweighting office pay off?)
- **Selection** $= w_b (r_{p,sector} - r_{b,sector})$ — did the specific names beat their sector?
- **Interaction** $= (w_p - w_b)(r_{p,sector} - r_{b,sector})$ — the cross-term.

This is the REPE-relevant decomposition: it separates *"we were in the right property types"* (allocation) from *"we picked the right REITs within each type"* (selection) — the exact question a real-estate investment committee asks.

In [9]:
# ── Cell 8: Contribution to return ───────────────────────────────────────────
def contribution_to_return(rets: pd.DataFrame, weights: pd.Series) -> pd.Series:
    """Per-asset contribution to the portfolio's geometric total return.

    Raw contribution = sum_t (w_i * r_it); the vector is then scaled so it
    sums exactly to the compounded portfolio return (adjusts for the
    arithmetic-vs-geometric gap in one proportional step).
    """
    port = rets.mul(weights, axis=1).sum(axis=1)
    geometric_total = float((1 + port).prod() - 1)
    raw = rets.mul(weights, axis=1).sum(axis=0)        # arithmetic contributions
    scale = geometric_total / raw.sum() if raw.sum() != 0 else 1.0
    return (raw * scale).sort_values(ascending=False)


contrib = contribution_to_return(simple_rets[PORT_TICKERS], W)
assert np.isclose(contrib.sum(), (1 + port_ret).prod() - 1, atol=1e-10), "contributions must close"

print("Contribution to 10-year portfolio return")
for tkr, c in contrib.items():
    print(f"  {tkr:<4} {c:+8.2%}   ({CFG.asset_class[tkr]})")
print(f"  {'SUM':<4} {contrib.sum():+8.2%}")

Contribution to 10-year portfolio return
  PLD   +34.65%   (Industrial)
  WELL  +27.22%   (Healthcare)
  EQIX  +26.77%   (Data Centers)
  EXR   +14.60%   (Storage)
  SPG   +13.38%   (Retail)
  AMT   +13.14%   (Towers)
  MAA   +11.16%   (Residential)
  O     +10.16%   (Retail)
  AVB    +7.56%   (Residential)
  BXP    +2.86%   (Office)
  SUM  +161.50%


In [10]:
# ── Cell 9: Brinson–Fachler attribution vs. policy benchmark ─────────────────
def brinson_attribution(
    rets: pd.DataFrame,
    port_weights: pd.Series,
    asset_class: dict[str, str],
    bench_class_weights: dict[str, float],
) -> pd.DataFrame:
    """Single-period Brinson–Fachler over the full horizon.

    Class returns are total (compounded) returns of the class baskets:
      - portfolio basket: tickers at *renormalized portfolio* weights
      - benchmark basket: same tickers at *equal* intra-class weights
        (a neutral 'own the class' benchmark), held at policy class weights.
    """
    classes = sorted(set(asset_class.values()))
    rows = []
    total_growth = lambda s: float((1 + s).prod() - 1)

    # Benchmark total return (needed by the allocation term)
    r_b_total = 0.0
    class_data = {}
    for cl in classes:
        tks = [t for t in port_weights.index if asset_class[t] == cl]
        w_p_cl = float(port_weights[tks].sum())
        w_b_cl = bench_class_weights[cl]
        # class return, portfolio construction (intra-class portfolio weights)
        intra_p = port_weights[tks] / port_weights[tks].sum()
        r_p_cl = total_growth(rets[tks].mul(intra_p, axis=1).sum(axis=1))
        # class return, benchmark construction (neutral equal weight)
        r_b_cl = total_growth(rets[tks].mean(axis=1))
        class_data[cl] = (w_p_cl, w_b_cl, r_p_cl, r_b_cl)
        r_b_total += w_b_cl * r_b_cl

    for cl, (w_p, w_b, r_p, r_b) in class_data.items():
        rows.append({
            "Class": cl,
            "Port Wgt": w_p, "Bench Wgt": w_b,
            "Port Ret": r_p, "Bench Ret": r_b,
            "Allocation":  (w_p - w_b) * (r_b - r_b_total),
            "Selection":   w_b * (r_p - r_b),
            "Interaction": (w_p - w_b) * (r_p - r_b),
        })

    df = pd.DataFrame(rows).set_index("Class")
    df["Total Effect"] = df[["Allocation", "Selection", "Interaction"]].sum(axis=1)
    df.loc["TOTAL"] = df.sum(numeric_only=True)
    df.loc["TOTAL", ["Port Ret", "Bench Ret"]] = np.nan   # class returns don't sum
    return df


brinson_df = brinson_attribution(
    simple_rets[PORT_TICKERS], W, CFG.asset_class, CFG.benchmark_class_weights
)
display(brinson_df.style.format("{:+.2%}", na_rep="—"))
print(f"\nActive return by property sector (Alloc + Select + Interact): "
      f"{brinson_df.loc['TOTAL', 'Total Effect']:+.2%}")

,Port Wgt,Bench Wgt,Port Ret,Bench Ret,Allocation,Selection,Interaction,Total Effect
Class,,,,,,,,
Data Centers,+13.00%,+12.50%,+218.64%,+218.64%,+0.34%,+0.00%,+0.00%,+0.34%
Healthcare,+10.00%,+12.50%,+351.75%,+351.75%,-5.04%,+0.00%,-0.00%,-5.04%
Industrial,+15.00%,+12.50%,+284.88%,+284.88%,+3.37%,+0.00%,+0.00%,+3.37%
Office,+8.00%,+12.50%,-22.03%,-22.03%,+7.75%,+0.00%,-0.00%,+7.75%
Residential,+17.00%,+12.50%,+71.55%,+70.41%,-3.59%,+0.14%,+0.05%,-3.39%
Retail,+18.00%,+12.50%,+76.84%,+78.37%,-3.95%,-0.19%,-0.08%,-4.22%
Storage,+9.00%,+12.50%,+131.50%,+131.50%,+0.65%,+0.00%,-0.00%,+0.65%
Towers,+10.00%,+12.50%,+87.56%,+87.56%,+1.56%,+0.00%,-0.00%,+1.56%
TOTAL,+100.00%,+100.00%,—,—,+1.10%,-0.05%,-0.03%,+1.02%



Active return by property sector (Alloc + Select + Interact): +1.02%


## 7 · Visualizations

Five institutional-standard charts, all interactive Plotly figures with a consistent color system. Each renders inline here **and** feeds the tabbed dashboard in the next section.

In [11]:
# ── Cell 10: Figure factory ──────────────────────────────────────────────────
COLORS = {"port": "#1f5fbf", "bench": "#9aa5b1", "neg": "#d64545", "pos": "#2f9e6e"}
LAYOUT = dict(font=dict(family="Inter, Helvetica, Arial", size=13),
              margin=dict(l=60, r=30, t=70, b=50), hovermode="x unified")


def fig_cumulative() -> go.Figure:
    """Growth of $10,000 — REIT portfolio vs. VNQ, with SPY for market context."""
    fig = go.Figure()
    fig.add_scatter(x=wealth_port.index, y=wealth_port, name="REIT Portfolio",
                    line=dict(color=COLORS["port"], width=2.2))
    fig.add_scatter(x=wealth_bench.index, y=wealth_bench, name="VNQ (REIT index)",
                    line=dict(color=COLORS["bench"], width=1.8, dash="dot"))
    fig.add_scatter(x=wealth_spy.index, y=wealth_spy, name="SPY (market)",
                    line=dict(color="#c9a227", width=1.4, dash="dash"))
    fig.update_layout(
        title="Growth of $10,000 — REIT Portfolio vs. VNQ vs. SPY",
        yaxis_title="Portfolio value ($)", **LAYOUT,
        updatemenus=[dict(type="buttons", x=1.0, y=1.15, xanchor="right",
            buttons=[dict(label="Linear", method="relayout", args=[{"yaxis.type": "linear"}]),
                     dict(label="Log",    method="relayout", args=[{"yaxis.type": "log"}])])])
    return fig


def fig_underwater() -> go.Figure:
    """Drawdown-from-peak with the max-DD trough annotated."""
    dd_p, dd_b = drawdown_series(port_ret), drawdown_series(bench_ret)
    mdd, peak, trough = max_drawdown(port_ret)
    fig = go.Figure()
    fig.add_scatter(x=dd_p.index, y=dd_p, name="REIT Portfolio", fill="tozeroy",
                    line=dict(color=COLORS["neg"], width=1.5),
                    fillcolor="rgba(214,69,69,0.25)")
    fig.add_scatter(x=dd_b.index, y=dd_b, name="VNQ",
                    line=dict(color=COLORS["bench"], width=1.2, dash="dot"))
    fig.add_annotation(x=trough, y=mdd, text=f"Max DD {mdd:.1%}<br>{trough.date()}",
                       showarrow=True, arrowhead=2, ay=40)
    fig.update_layout(title="Underwater Plot — Drawdown from Peak",
                      yaxis_tickformat=".0%", yaxis_title="Drawdown", **LAYOUT)
    return fig


def fig_rolling() -> go.Figure:
    """Dual panel: rolling 90d volatility (top) and rolling 1y Sharpe (bottom)."""
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.09,
                        subplot_titles=(f"Rolling {ROLL_VOL_WINDOW}-Day Annualized Volatility",
                                        f"Rolling {ROLL_SHARPE_WINDOW}-Day Sharpe Ratio"))
    fig.add_scatter(x=roll_vol_port.index, y=roll_vol_port, name="Portfolio vol",
                    line=dict(color=COLORS["port"]), row=1, col=1)
    fig.add_scatter(x=roll_vol_bench.index, y=roll_vol_bench, name="SPY vol",
                    line=dict(color=COLORS["bench"], dash="dot"), row=1, col=1)
    fig.add_scatter(x=roll_sharpe_port.index, y=roll_sharpe_port, name="Portfolio Sharpe",
                    line=dict(color=COLORS["port"]), row=2, col=1)
    fig.add_scatter(x=roll_sharpe_bench.index, y=roll_sharpe_bench, name="SPY Sharpe",
                    line=dict(color=COLORS["bench"], dash="dot"), row=2, col=1)
    fig.add_hline(y=0, line=dict(color="#999", width=1), row=2, col=1)
    fig.update_yaxes(tickformat=".0%", row=1, col=1)
    fig.update_layout(title="Rolling Risk Diagnostics", height=650, **LAYOUT)
    return fig


def fig_waterfall() -> go.Figure:
    """Contribution-to-return waterfall, sorted best to worst."""
    fig = go.Figure(go.Waterfall(
        x=list(contrib.index) + ["Portfolio"],
        y=list(contrib.values) + [0.0],
        measure=["relative"] * len(contrib) + ["total"],
        text=[f"{v:+.1%}" for v in contrib] + [f"{contrib.sum():+.1%}"],
        textposition="outside",
        increasing=dict(marker_color=COLORS["pos"]),
        decreasing=dict(marker_color=COLORS["neg"]),
        totals=dict(marker_color=COLORS["port"]),
        connector=dict(line=dict(color="#cbd2d9")),
    ))
    fig.update_layout(title=f"Contribution to {CFG.years}-Year Total Return by Asset",
                      yaxis_tickformat=".0%", **LAYOUT)
    return fig


def fig_brinson() -> go.Figure:
    """Grouped bars: allocation / selection / interaction per asset class."""
    b = brinson_df.drop(index="TOTAL")
    fig = go.Figure()
    for effect, color in [("Allocation", COLORS["port"]),
                          ("Selection", COLORS["pos"]),
                          ("Interaction", COLORS["bench"])]:
        fig.add_bar(x=b.index, y=b[effect], name=effect, marker_color=color,
                    text=[f"{v:+.1%}" for v in b[effect]], textposition="outside")
    fig.update_layout(title="Brinson–Fachler Attribution by Property Sector",
                      barmode="group", yaxis_tickformat=".0%", **LAYOUT)
    return fig


def fig_corr() -> go.Figure:
    """Full-period correlation heatmap, ordered by property sector."""
    order = sorted(PORT_TICKERS, key=lambda t: (CFG.property_sector[t], t))
    corr = simple_rets[order].corr()
    fig = go.Figure(go.Heatmap(
        z=corr.values, x=corr.columns, y=corr.index,
        colorscale="RdBu", zmid=0, zmin=-1, zmax=1,
        text=corr.round(2).values, texttemplate="%{text}",
        colorbar=dict(title="ρ"),
    ))
    fig.update_layout(title="REIT Daily Return Correlation Matrix (10-Year)",
                      height=620, **{k: v for k, v in LAYOUT.items() if k != "hovermode"})
    fig.update_yaxes(autorange="reversed")
    return fig


def fig_rate_sensitivity(window: int = ROLL_VOL_WINDOW) -> go.Figure:
    """THE star chart: rolling correlation of REITs (VNQ) with long Treasuries (TLT).

    Tells the rate-sensitivity story — REITs decoupled from bonds in the ZIRP
    era, then re-coupled hard through the 2022 rate shock as both sold off.
    """
    roll_corr = bench_ret.rolling(window).corr(tlt_ret).dropna()
    port_corr = port_ret.rolling(window).corr(tlt_ret).dropna()
    fig = go.Figure()
    fig.add_scatter(x=roll_corr.index, y=roll_corr, name="VNQ vs. TLT",
                    line=dict(color=COLORS["port"], width=2))
    fig.add_scatter(x=port_corr.index, y=port_corr, name="REIT Portfolio vs. TLT",
                    line=dict(color=COLORS["pos"], width=1.6, dash="dot"))
    fig.add_hline(y=0, line=dict(color="#999", width=1))
    fig.add_vrect(x0="2022-01-01", x1="2023-10-31",
                  fillcolor="rgba(214,69,69,0.10)", line_width=0,
                  annotation_text="2022–23 rate shock", annotation_position="top left")
    fig.update_layout(
        title=f"Rate Sensitivity — Rolling {window}-Day Correlation: REITs vs. Long Treasuries",
        yaxis_title="Correlation", yaxis_range=[-1, 1], **LAYOUT)
    return fig


FIGS = {
    "Performance":      [fig_cumulative()],
    "Risk":             [fig_underwater(), fig_rolling()],
    "Attribution":      [fig_waterfall(), fig_brinson()],
    "Rate Sensitivity": [fig_rate_sensitivity(), fig_corr()],
}
# Render everything inline for the notebook reader
for tab, figs in FIGS.items():
    for f in figs:
        f.show()

## 8 · Tabbed HTML Dashboard

Assembles the six figures into a single **standalone** `dashboard.html` — pure HTML/CSS/JS tabs, Plotly loaded from CDN. No server, no dependencies: double-click it, email it, or host it on GitHub Pages.

In [12]:
# ── Cell 11: Assemble standalone tabbed dashboard ────────────────────────────
from datetime import date

def build_dashboard(figs_by_tab: dict[str, list], path: str = "dashboard.html") -> str:
    """Write a self-contained tabbed HTML dashboard and return its path."""
    css = """
    body{font-family:Inter,Helvetica,Arial,sans-serif;margin:0;background:#f4f6f8}
    header{background:#102a43;color:#fff;padding:22px 36px}
    header h1{margin:0;font-size:22px} header p{margin:6px 0 0;color:#bcccdc;font-size:13px}
    .tabs{display:flex;gap:4px;background:#102a43;padding:0 36px}
    .tabs button{border:0;padding:12px 22px;font-size:14px;cursor:pointer;
                 background:#243b53;color:#d9e2ec;border-radius:8px 8px 0 0}
    .tabs button.active{background:#f4f6f8;color:#102a43;font-weight:600}
    .panel{display:none;padding:24px 36px} .panel.active{display:block}
    .card{background:#fff;border-radius:10px;box-shadow:0 1px 4px rgba(16,42,67,.12);
          padding:12px;margin-bottom:24px}
    """
    js = """
    function showTab(i){
      document.querySelectorAll('.panel').forEach((p,j)=>p.classList.toggle('active',i===j));
      document.querySelectorAll('.tabs button').forEach((b,j)=>b.classList.toggle('active',i===j));
      window.dispatchEvent(new Event('resize'));  // force Plotly re-layout on reveal
    }
    """
    buttons, panels = [], []
    for i, (name, figs) in enumerate(figs_by_tab.items()):
        active = " active" if i == 0 else ""
        buttons.append(f'<button class="{active.strip()}" onclick="showTab({i})">{name}</button>')
        charts = "".join(
            '<div class="card">' +
            f.to_html(full_html=False, include_plotlyjs=False,
                      config={"displaylogo": False, "responsive": True}) +
            "</div>"
            for f in figs
        )
        panels.append(f'<div class="panel{active}">{charts}</div>')

    html = (
        "<!DOCTYPE html><html><head><meta charset='utf-8'>"
        "<title>REIT Sector Performance & Rate Sensitivity</title>"
        "<script src='https://cdn.plot.ly/plotly-2.32.0.min.js'></script>"
        f"<style>{css}</style><script>{js}</script></head><body>"
        "<header><h1>REIT Sector Performance &amp; Rate-Sensitivity Dashboard</h1>"
        f"<p>{CFG.years}-year daily history · generated {date.today()} · "
        f"benchmark VNQ · 10 REITs across 8 property sectors · risk-free 3M T-bill</p></header>"
        f"<div class='tabs'>{''.join(buttons)}</div>"
        f"{''.join(panels)}</body></html>"
    )
    with open(path, "w", encoding="utf-8") as fh:
        fh.write(html)
    return path


dash_path = build_dashboard(FIGS)
print(f"Dashboard written -> {dash_path}  "
      "(in Colab: Files pane -> download, or serve via files.download)")

# Optional: trigger a browser download when running in Colab
try:
    from google.colab import files
    files.download(dash_path)
except Exception:
    pass  # not running in Colab — file is on local disk

Dashboard written -> dashboard.html  (in Colab: Files pane -> download, or serve via files.download)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 9 · Exports & Executive Summary

Writes the three CSVs a fund administrator would archive, then prints a plain-English summary — the paragraph that would open the client letter.

In [13]:
# ── Cell 12: CSV exports + executive summary ─────────────────────────────────
metrics_df.to_csv("metrics_summary.csv")
contrib.rename("Contribution").to_frame().assign(
    AssetClass=lambda d: d.index.map(CFG.asset_class)
).to_csv("contribution_by_asset.csv")
brinson_df.to_csv("brinson_attribution.csv")
print("Saved: metrics_summary.csv, contribution_by_asset.csv, brinson_attribution.csv\n")

p, b = rows["REIT Portfolio"], rows["Benchmark (VNQ)"]
best, worst = contrib.index[0], contrib.index[-1]
active = brinson_df.loc["TOTAL", "Total Effect"]

print("=" * 74)
print("EXECUTIVE SUMMARY")
print("=" * 74)
print(f"Over the {CFG.years}-year window the REIT portfolio compounded at "
      f"{p['CAGR']:.1%} annually ({p['Total Return']:+.0%} cumulative) with "
      f"{p['Ann. Vol']:.1%} volatility, a Sharpe of {p['Sharpe']:.2f} "
      f"(vs. {b['Sharpe']:.2f} for VNQ) and a maximum drawdown of "
      f"{p['Max DD']:.1%} (VNQ: {b['Max DD']:.1%}).")
print(f"\n{best} ({CFG.property_sector[best]}) was the largest contributor "
      f"({contrib[best]:+.1%}); {worst} ({CFG.property_sector[worst]}) the "
      f"largest detractor ({contrib[worst]:+.1%}).")
print(f"\nAgainst an equal-sector-weight policy benchmark, sector allocation and "
      f"security selection added {active:+.1%} of active return — see the "
      f"Attribution tab for the property-sector breakdown.")
print(f"\nRate sensitivity: the portfolio's daily correlation to long Treasuries "
      f"(TLT) over the full period is {p['Corr vs TLT']:+.2f}; the Rate-Sensitivity "
      f"tab shows how that correlation spiked during the 2022–23 rate shock.")

Saved: metrics_summary.csv, contribution_by_asset.csv, brinson_attribution.csv

EXECUTIVE SUMMARY
Over the 10-year window the REIT portfolio compounded at 10.1% annually (+162% cumulative) with 21.7% volatility, a Sharpe of 0.45 (vs. 0.23 for VNQ) and a maximum drawdown of -40.4% (VNQ: -42.4%).

PLD (Industrial) was the largest contributor (+34.7%); BXP (Office) the largest detractor (+2.9%).

Against an equal-sector-weight policy benchmark, sector allocation and security selection added +1.0% of active return — see the Attribution tab for the property-sector breakdown.

Rate sensitivity: the portfolio's daily correlation to long Treasuries (TLT) over the full period is +0.06; the Rate-Sensitivity tab shows how that correlation spiked during the 2022–23 rate shock.


## Key Findings

**Risk-adjusted outperformance.** Over the 10-year window the portfolio compounded at
10.1% annually (+162% cumulative) versus 5.0% for the REIT index (VNQ) — roughly double
the return, with a Sharpe of 0.45 vs. 0.23. The edge came from return rather than lower
risk: portfolio volatility (21.7%) slightly exceeded VNQ's (20.7%), reflecting a
concentrated 10-name book.

**Property-type divergence drove results.** Returns dispersed enormously by property
sector — Welltower (healthcare, +16.3% CAGR), Prologis (industrial, +14.5%), and Equinix
(data centers, +12.3%) led, while BXP (office) was the only holding with a negative
10-year return (−2.5% CAGR). The overweight to structural winners (industrial, data
centers, towers) and underweight to office is the portfolio's central thesis.

**Attribution.** Active return of +1.0% vs. an equal-sector-weight policy benchmark came
almost entirely from sector allocation (+1.1%) rather than security selection (−0.05%).
The strongest single call was underweighting office (+7.7% relative; the sector returned
−22%); the largest drag was underweighting healthcare (−5.0%), which was in fact the
top-performing sector. Selection is measured only in the two multi-name sectors
(Residential and Retail); the other six hold a single REIT each.

**Drawdown.** Maximum drawdown was −40.4% during the Feb–Mar 2020 COVID crash, marginally
shallower than VNQ's −42.4% — the sector tilts offered limited downside protection in a
correlated sell-off.

**Rate sensitivity.** Daily correlation to long Treasuries (TLT) averaged just +0.06 over
the full decade, but the rolling 90-day correlation reveals a clear regime shift: it ran
near zero to negative through 2020–21 (bottoming near −0.5 in the COVID crash), then
turned persistently positive — climbing to roughly +0.6 and holding at +0.5 to +0.6 from
late 2023 onward. As long-end rates rose, REITs and long bonds began moving together,
confirming that listed real estate re-couples to duration when the cost of capital moves —
the core macro driver of the asset class.
